In [ ]:
import wikipediaapi
wiki_wiki = wikipediaapi.Wikipedia(user_agent='PyrMyd (kalebmokua@gmail.com)', language='en')

page_py = wiki_wiki.page('Python_(programming_language)')

In [ ]:
print(f"{page_py.title}: {page_py.summary[0:60]}...")

In [ ]:
print("Page - Exists: %s" % page_py.exists())

In [ ]:
wiki_wiki = wikipediaapi.Wikipedia(
    user_agent='PyrMyd (kalebmokua@gmail.com)',
    language='en',
    extract_format=wikipediaapi.ExtractFormat.WIKI
)

p_wiki = wiki_wiki.page("Python_(programming_language)")
print(p_wiki.text)

In [2]:
import wikipediaapi

wiki = wikipediaapi.Wikipedia(
    user_agent="PyrMyd (kalebmokua@gmail.com)",  # required, must have contact info
    language="en"
)

def wikipedia_search_to_docs(query: str, max_docs: int = 2):
    """Mimics WikipediaLoader(query=..., load_max_docs=...).load() output shape."""
    results = wiki.search(query, limit=max_docs)
    docs = []
    for title, page in list(results.pages.items())[:max_docs]:
        docs.append({
            "page_content": page.text,   # or page.text for full article
            "metadata": {
                "title": page.title,
                "source": page.fullurl,
            }
        })
    return docs

wikipedia_search_to_docs("LangGraph architectural advantages large-scale enterprise deployments existing infrastructure compliance requirements", max_docs=2)

[]

In [ ]:
import wikipediaapi
import difflib

wiki = wikipediaapi.Wikipedia(
    user_agent="PyrMyd (kalebmokua@gmail.com)",
    language="en"
)

def _is_relevant(query: str, title: str, threshold: float = 0.35) -> bool:
    """Cheap guard against fuzzy-search noise (e.g. 'LangGraph' -> 'Java')."""
    query_words = set(query.lower().split())
    title_words = set(title.lower().replace("(", "").replace(")", "").split())
    overlap = query_words & title_words
    similarity = difflib.SequenceMatcher(None, query.lower(), title.lower()).ratio()
    return bool(overlap) or similarity >= threshold

def wikipedia_search_to_docs(query: str, max_docs: int = 2):
    docs = []

    # 1. Try exact title match first — most precise, no fuzziness involved
    exact_page = wiki.page(query)
    if exact_page.exists():
        docs.append({
            "page_content": exact_page.summary,
            "metadata": {"title": exact_page.title, "source": exact_page.fullurl},
        })

    # 2. Fall back to search, but filter out irrelevant fuzzy matches
    if len(docs) < max_docs:
        results = wiki.search(query, limit=max_docs * 3)  # over-fetch, then filter
        for title, page in results.pages.items():
            if len(docs) >= max_docs:
                break
            if any(d["metadata"]["title"] == page.title for d in docs):
                continue  # avoid duplicate of the exact match
            if not _is_relevant(query, page.title):
                continue
            docs.append({
                "page_content": page.summary,
                "metadata": {"title": page.title, "source": page.fullurl},
            })

    return docs

wikipedia_search_to_docs("LangGraph", max_docs=2)